# Steering Continuous Reasoning via Latent Intervention
### Phases 1–3: Train → Extract Truth Vector → Steer

**Before running:** Set runtime to T4 GPU → `Runtime → Change runtime type → T4 GPU`

| Phase | Script | Est. Time (T4) |
|-------|--------|----------------|
| 1 | Base CODI training | ~2 hours |
| 2 | Truth vector extraction | ~10 min |
| 3 | Steering sweep (6 alphas) | ~20 min |


## 🔧 Cell 1: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {name}  ({mem:.1f} GB)')
else:
    print('✗ No GPU detected!')
    print('  Go to Runtime → Change runtime type → T4 GPU')

## 📦 Cell 2: Clone Repository

In [ ]:
import os

REPO = 'https://github.com/nabilanewaz/TokenSkip.git'
REPO_DIR = '/content/TokenSkip'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes...')
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO} {REPO_DIR}
    %cd {REPO_DIR}

!echo "\nRepository contents:"
!ls -la

## 📋 Cell 3: Install Dependencies

In [ ]:
%%capture install_output
# Pin exact versions CODI requires
!pip install --force-reinstall --no-deps \
    peft==0.15.2 \
    datasets==3.6.0 \
    huggingface_hub \
    transformers==4.52.4 \
    accelerate==1.7.0 \
    safetensors

!pip install -q \
    peft==0.15.2 \
    datasets==3.6.0 \
    huggingface_hub \
    transformers==4.52.4 \
    accelerate==1.7.0 \
    safetensors

print('✓ Dependencies installed')

## ✅ Cell 4: Verify Files and Data

In [ ]:
import os, pathlib

required_files = [
    'phase1_train.py',
    'phase2_extract_vector.py',
    'phase3_steer_inference.py',
    'codi_bundle/train.py',
    'codi_bundle/src/model.py',
    'datasets/gsm8k_split/llm_train.jsonl',
    'datasets/gsm8k_split/steer_train.jsonl',
    'datasets/gsm8k_split/validation.jsonl',
    'datasets/gsm8k_split/test.jsonl',
]

all_ok = True
for f in required_files:
    p = pathlib.Path(f)
    if p.exists():
        size = p.stat().st_size
        if p.suffix == '.jsonl':
            n = sum(1 for _ in open(p))
            print(f'  ✓ {f}  ({n} examples)')
        else:
            print(f'  ✓ {f}  ({size/1024:.1f} KB)')
    else:
        print(f'  ✗ MISSING: {f}')
        all_ok = False

if all_ok:
    print('\n✓ All files present. Ready to train!')
else:
    print('\n✗ Some files missing. Check your git push included all new files.')

---
## 🚀 PHASE 1: Base Model Training
Trains CODI-GPT2 on `llm_train.jsonl` using curriculum learning.
- Stage A: discrete CoT tokens (grounds reasoning)
- Stage B: continuous latent vectors (Coconut/CODI style)

**~2 hours on T4 GPU**

In [ ]:
!python phase1_train.py \
    --train-data datasets/gsm8k_split/llm_train.jsonl \
    --val-data   datasets/gsm8k_split/validation.jsonl \
    --output-dir outputs/phase1_checkpoint \
    --num_epochs 3 \
    --batch_size 4 \
    --learning_rate 2e-4 \
    --bf16

In [ ]:
# Monitor training progress (run this cell repeatedly while Cell above is running)
import glob, os

log_path = 'outputs/phase1_checkpoint/train_log.txt'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f'Log has {len(lines)} lines. Last 20:')
    print(''.join(lines[-20:]))
else:
    print('Log not created yet — training may still be initializing...')

In [ ]:
# Verify Phase 1 checkpoint was saved
import glob

ckpts = glob.glob('outputs/phase1_checkpoint/**/*.safetensors', recursive=True) + \
        glob.glob('outputs/phase1_checkpoint/**/*.bin', recursive=True)

if ckpts:
    print(f'✓ Phase 1 complete! Found {len(ckpts)} checkpoint file(s):')
    for c in ckpts:
        size_mb = os.path.getsize(c) / 1e6
        print(f'    {c}  ({size_mb:.1f} MB)')
else:
    print('✗ No checkpoint found. Check train_log.txt for errors.')

meta_path = 'outputs/phase1_checkpoint/phase1_metadata.json'
if os.path.exists(meta_path):
    import json
    meta = json.load(open(meta_path))
    print(f'\nTraining metadata:')
    print(f'  n_train   : {meta["n_train"]}')
    print(f'  epochs    : {meta["epochs"]}')
    print(f'  elapsed   : {meta["elapsed_hours"]:.2f}h')

---
## 🧠 PHASE 2: Truth Vector Extraction
Runs the Phase 1 model on `steer_train.jsonl`, classifies traces as H+ (correct) / H- (wrong),
then computes `v_truth = mean(H+) - mean(H-)` per latent step.

**~10 minutes on T4 GPU**

In [ ]:
!python phase2_extract_vector.py \
    --steer-data datasets/gsm8k_split/steer_train.jsonl \
    --ckpt-dir   outputs/phase1_checkpoint \
    --out-dir    outputs/phase2_truth_vector \
    --n-samples  5 \
    --bf16

In [ ]:
# Inspect Phase 2 results
import json, os

stats_path = 'outputs/phase2_truth_vector/stats.json'
if os.path.exists(stats_path):
    stats = json.load(open(stats_path))
    print('Phase 2 Truth Vector Stats')
    print('=' * 40)
    print(f'  H+ (correct traces) : {stats["n_pos"]}')
    print(f'  H- (wrong traces)   : {stats["n_neg"]}')
    print(f'  Balance (H+/all)    : {stats["balance_ratio"]:.1%}')
    print(f'  Vector dim D        : {stats["D"]}')
    print(f'  Num latent steps L  : {stats["L"]}')
    print(f'  Global ||v_truth||  : {stats["v_truth_global_norm"]:.4f}')
    print(f'  Per-step norms      : {stats["v_truth_per_step_norms"]}')
    print(f'  Sigma per step      : {stats["sigma_per_step"]}')
else:
    print('✗ Stats not found. Phase 2 may not have completed.')

---
## 🎯 PHASE 3: Inference-Time Steering Sweep
Sweeps α ∈ {0.0, 0.1, 0.5, 1.0, 2.0, 5.0} and evaluates on the held-out test set.

Measures:
- **Accuracy** per alpha
- **Flip Rate** — % of baseline failures corrected by steering
- **Cosine Similarity** — trajectory alignment with v_truth

**~20 minutes on T4 GPU**

In [ ]:
!python phase3_steer_inference.py \
    --eval-data  datasets/gsm8k_split/test.jsonl \
    --vector-dir outputs/phase2_truth_vector \
    --ckpt-dir   outputs/phase1_checkpoint \
    --out-dir    outputs/phase3_results \
    --alphas 0.0 0.1 0.5 1.0 2.0 5.0 \
    --bf16

In [ ]:
# Display full results summary
import json, os

summary_path = 'outputs/phase3_results/summary.json'
if not os.path.exists(summary_path):
    print('Summary not found yet.')
else:
    summary = json.load(open(summary_path))
    results  = sorted(summary['results'],       key=lambda r: r['alpha'])
    flips    = {f['alpha']: f for f in summary.get('flip_analysis', [])}
    baseline_acc = next((r['accuracy'] for r in results if r['alpha'] == 0.0), None)

    print('STEERING RESULTS')
    print('=' * 75)
    print(f'{"α":>6}  {"Accuracy":>10}  {"Δ baseline":>11}  {"Flip Rate":>10}  {"Cos(h,v)":>9}')
    print('-' * 75)
    for r in results:
        a    = r['alpha']
        acc  = r['accuracy']
        cos  = r['mean_cosine_global']
        delta = (acc - baseline_acc) if baseline_acc else 0
        f    = flips.get(a, {})
        fr   = f'{f["flip_rate"]:.1%}' if f.get('flip_rate') is not None else '  baseline'
        marker = '  ← control' if a == 0.0 else ''
        print(f'{a:>6.1f}  {acc*100:>9.2f}%  {delta*100:>+10.2f}%  {fr:>10}  {cos:>9.4f}{marker}')
    print('=' * 75)

    best = max(results, key=lambda r: r['accuracy'])
    print(f'\nBest α = {best["alpha"]}  (accuracy: {best["accuracy"]:.2%})')
    if baseline_acc:
        print(f'Gain over baseline: {(best["accuracy"] - baseline_acc)*100:+.2f}%')

In [ ]:
# Plot accuracy vs alpha
import json
import matplotlib.pyplot as plt

summary = json.load(open('outputs/phase3_results/summary.json'))
results = sorted(summary['results'], key=lambda r: r['alpha'])

alphas   = [r['alpha']          for r in results]
accs     = [r['accuracy'] * 100 for r in results]
cosines  = [r['mean_cosine_global'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy vs Alpha
ax = axes[0]
ax.plot(alphas, accs, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axhline(accs[0], color='gray', linestyle='--', label=f'Baseline ({accs[0]:.1f}%)')
ax.set_xlabel('Steering Strength α', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Steering Strength', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
for a, acc in zip(alphas, accs):
    ax.annotate(f'{acc:.1f}%', (a, acc), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)

# Cosine Similarity vs Alpha
ax = axes[1]
ax.plot(alphas, cosines, 's-', color='darkorange', linewidth=2, markersize=8)
ax.set_xlabel('Steering Strength α', fontsize=12)
ax.set_ylabel('Mean Cosine(h_t, v_truth)', fontsize=12)
ax.set_title('Trajectory Alignment with Truth Vector', fontsize=13)
ax.grid(True, alpha=0.3)

plt.suptitle('Steering Continuous Reasoning via Latent Intervention', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/phase3_results/steering_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → outputs/phase3_results/steering_plot.png')

---
## 💾 Save Everything to Google Drive (Optional but Recommended)

In [ ]:
# Mount Google Drive to avoid losing work when session ends
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TokenSkip_Results'
!mkdir -p {DRIVE_DIR}

!cp -r outputs/phase1_checkpoint {DRIVE_DIR}/
!cp -r outputs/phase2_truth_vector {DRIVE_DIR}/
!cp -r outputs/phase3_results {DRIVE_DIR}/

print(f'✓ All outputs backed up to {DRIVE_DIR}/')

In [ ]:
# Or download as a zip
!zip -r results.zip \
    outputs/phase1_checkpoint \
    outputs/phase2_truth_vector \
    outputs/phase3_results

from google.colab import files
files.download('results.zip')
print('✓ Download started')